# Carbon-24 Feature Extraction v2

Nâng cấp so với v1:
- Thêm `lattice_anisotropy`, `angle_std`, `bond_length_range`
- Thêm `fraction_sp2`, `fraction_sp3`, `fraction_sp` (hybridization)
- Thêm `packing_fraction` (ước lượng)
- Sửa `symprec=0.01` (chính xác hơn, fallback về 0.1 nếu lỗi)
- Sửa `bond_cutoff=2.0` Å (bắt đủ liên kết C–C dài)
- `relative_energy` tính chỉ trên tập train (tránh data leakage)
- Thêm báo cáo phân phối hybridization và kiểm tra bond_cutoff

## 1. Cài đặt thư viện

In [ ]:
!pip install -q pymatgen datasets seaborn scikit-learn scipy tqdm

## 2. Import

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

pd.set_option('display.max_columns', None)
print('Imports OK')

## 3. Load dataset

In [ ]:
ds = load_dataset('albertvillanova/carbon_24')
print(ds)

In [ ]:
dfs = []
for split in ds.keys():
    tmp = ds[split].to_pandas()
    tmp['split'] = split
    tmp['source_file'] = 'huggingface'
    dfs.append(tmp)

carbon_df = pd.concat(dfs, ignore_index=True)
print('carbon_df shape:', carbon_df.shape)
print('columns:', carbon_df.columns.tolist())
display(carbon_df.head(3))

## 4. Thiết lập thư mục

In [ ]:
PROJECT_DIR  = '/kaggle/working/carbon24_project_v2'
DATA_DIR     = f'{PROJECT_DIR}/data'
CHECKPT_DIR  = f'{PROJECT_DIR}/checkpoints'

for d in [DATA_DIR, CHECKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Directories ready.')

## 5. Xác định cột CIF và energy

In [ ]:
# Tự động phát hiện cột CIF
cif_col = next(
    (col for col in carbon_df.columns
     if carbon_df[col].dropna().astype(str).head(20)
        .str.contains('_cell_length|_atom_site|data_').any()),
    None
)

# Tự động phát hiện cột energy
energy_col = next(
    (col for col in carbon_df.columns
     if any(kw in col.lower() for kw in ['energy', 'formation', 'enthalpy'])),
    None
)

print('cif_col   :', cif_col)
print('energy_col:', energy_col)

## 6. Kiểm tra bond_cutoff trên mẫu nhỏ

Trước khi chạy toàn bộ, kiểm tra phân phối `max_bond_length` trên 200 cấu trúc để xác nhận cutoff 2.0 Å là đủ.

In [ ]:
PROBE_CUTOFF = 2.5   # cutoff rộng để thăm dò
probe_max_bonds = []

for _, row in tqdm(carbon_df.sample(200, random_state=42).iterrows(),
                   total=200, desc='Probing bond lengths'):
    try:
        st = Structure.from_str(row[cif_col], fmt='cif')
        for site in st:
            neighs = st.get_neighbors(site, PROBE_CUTOFF)
            dists = [n.nn_distance for n in neighs if n.nn_distance > 0.5]
            if dists:
                probe_max_bonds.append(max(dists))
    except Exception:
        pass

probe_series = pd.Series(probe_max_bonds)
print('Bond length distribution (probe cutoff=2.5 Å):')
print(probe_series.describe())
print(f'\n95th percentile: {probe_series.quantile(0.95):.4f} Å')
print(f'99th percentile: {probe_series.quantile(0.99):.4f} Å')
print('\n=> Nếu 99th percentile < 2.0 Å thì cutoff=2.0 là an toàn.')

In [ ]:
# Đặt cutoff dựa trên kết quả thăm dò
# Mặc định 2.0 Å — đủ cho C-C sp/sp2/sp3 (max ~1.54 Å) với biên an toàn
BOND_CUTOFF = 2.0
SYMPREC_STRICT = 0.01   # chính xác hơn v1 (0.1)
SYMPREC_LOOSE  = 0.1    # fallback nếu strict thất bại

print(f'BOND_CUTOFF  = {BOND_CUTOFF} Å')
print(f'SYMPREC      = {SYMPREC_STRICT} (fallback {SYMPREC_LOOSE})')

## 7. Hàm trích xuất đặc trưng (v2)

### Đặc trưng mới so với v1

| Đặc trưng | Mô tả |
|---|---|
| `lattice_anisotropy` | max(a,b,c) / min(a,b,c) — đo mức độ dị hướng mạng |
| `angle_std` | std(alpha, beta, gamma) — đo biến dạng góc |
| `bond_length_range` | max - min bond length — đo độ đồng đều liên kết |
| `fraction_sp` | tỉ lệ nguyên tử có coordination=2 (sp, carbyne) |
| `fraction_sp2` | tỉ lệ nguyên tử có coordination=3 (sp², graphene) |
| `fraction_sp3` | tỉ lệ nguyên tử có coordination=4 (sp³, diamond) |
| `packing_fraction` | ước lượng mật độ đóng gói dựa trên bán kính C |
| `symprec_used` | giá trị symprec thực sự dùng (strict hay loose) |

In [ ]:
# Bán kính cộng hóa trị của Carbon (Å)
C_RADIUS = 0.77

def extract_features_v2(
    cif_str,
    symprec_strict=SYMPREC_STRICT,
    symprec_loose=SYMPREC_LOOSE,
    bond_cutoff=BOND_CUTOFF,
    compute_symmetry=True,
    compute_bonding=True
):
    """
    Trích xuất đặc trưng từ chuỗi CIF của cấu trúc carbon.
    Trả về dict với đầy đủ đặc trưng hình học, đối xứng, liên kết và hybridization.
    """
    structure = Structure.from_str(cif_str, fmt='cif')
    lattice   = structure.lattice

    # ── Thông số cơ bản ──────────────────────────────────────────────────────
    num_atoms       = len(structure)
    volume          = float(structure.volume)
    volume_per_atom = volume / num_atoms if num_atoms > 0 else np.nan
    density         = float(structure.density)

    # ── Thông số mạng tinh thể ───────────────────────────────────────────────
    a     = float(lattice.a)
    b     = float(lattice.b)
    c     = float(lattice.c)
    alpha = float(lattice.alpha)
    beta  = float(lattice.beta)
    gamma = float(lattice.gamma)

    b_over_a = b / a if a != 0 else np.nan
    c_over_a = c / a if a != 0 else np.nan

    # Tổng độ lệch góc so với 90° (v1)
    angle_deviation = abs(alpha - 90) + abs(beta - 90) + abs(gamma - 90)

    # [MỚI] Std của 3 góc — đo biến dạng góc đồng đều hơn
    angle_std = float(np.std([alpha, beta, gamma]))

    # [MỚI] Dị hướng mạng — phân biệt cấu trúc layered vs isotropic
    lattice_anisotropy = max(a, b, c) / min(a, b, c) if min(a, b, c) != 0 else np.nan

    # ── Đối xứng ─────────────────────────────────────────────────────────────
    space_group_number = np.nan
    space_group_symbol = 'unknown'
    crystal_system     = 'unknown'
    symprec_used       = np.nan

    if compute_symmetry:
        # Thử strict trước, fallback về loose
        for sp in [symprec_strict, symprec_loose]:
            try:
                sga = SpacegroupAnalyzer(structure, symprec=sp)
                space_group_number = sga.get_space_group_number()
                space_group_symbol = sga.get_space_group_symbol()
                crystal_system     = sga.get_crystal_system()
                symprec_used       = sp
                break
            except Exception:
                continue

    # ── Liên kết và coordination ─────────────────────────────────────────────
    bond_lengths = []
    coord_nums   = []

    if compute_bonding:
        try:
            for site in structure:
                neighs = structure.get_neighbors(site, bond_cutoff)
                # Lọc khoảng cách hợp lý: > 0.5 Å (loại self-image)
                dists = [float(n.nn_distance) for n in neighs
                         if 0.5 < n.nn_distance <= bond_cutoff]
                coord_nums.append(len(dists))
                bond_lengths.extend(dists)
        except Exception:
            bond_lengths = []
            coord_nums   = []

    if bond_lengths:
        mean_bond_length  = float(np.mean(bond_lengths))
        std_bond_length   = float(np.std(bond_lengths))
        min_bond_length   = float(np.min(bond_lengths))
        max_bond_length   = float(np.max(bond_lengths))
        # [MỚI] Khoảng biến thiên độ dài liên kết
        bond_length_range = max_bond_length - min_bond_length
    else:
        mean_bond_length  = np.nan
        std_bond_length   = np.nan
        min_bond_length   = np.nan
        max_bond_length   = np.nan
        bond_length_range = np.nan

    if coord_nums:
        mean_coordination = float(np.mean(coord_nums))
        std_coordination  = float(np.std(coord_nums))
        min_coordination  = float(np.min(coord_nums))
        max_coordination  = float(np.max(coord_nums))

        # [MỚI] Phân loại hybridization theo coordination number
        # sp  (linear, carbyne)  : coord = 2
        # sp2 (graphene, fullerene): coord = 3
        # sp3 (diamond, lonsdaleite): coord = 4
        n_total  = len(coord_nums)
        n_sp     = sum(1 for c in coord_nums if c == 2)
        n_sp2    = sum(1 for c in coord_nums if c == 3)
        n_sp3    = sum(1 for c in coord_nums if c == 4)
        n_other  = n_total - n_sp - n_sp2 - n_sp3

        fraction_sp    = n_sp    / n_total
        fraction_sp2   = n_sp2   / n_total
        fraction_sp3   = n_sp3   / n_total
        fraction_other = n_other / n_total
    else:
        mean_coordination = np.nan
        std_coordination  = np.nan
        min_coordination  = np.nan
        max_coordination  = np.nan
        fraction_sp       = np.nan
        fraction_sp2      = np.nan
        fraction_sp3      = np.nan
        fraction_other    = np.nan

    # [MỚI] Packing fraction: thể tích cầu C / thể tích ô mạng
    # Dùng bán kính cộng hóa trị C = 0.77 Å
    vol_atoms       = num_atoms * (4/3) * np.pi * (C_RADIUS ** 3)
    packing_fraction = vol_atoms / volume if volume > 0 else np.nan

    formula  = structure.composition.reduced_formula
    elements = [str(el) for el in structure.composition.elements]

    return {
        # Định danh
        'formula'  : formula,
        'elements' : ','.join(elements),
        'num_atoms': int(num_atoms),

        # Thông số mạng
        'a'    : a,
        'b'    : b,
        'c'    : c,
        'alpha': alpha,
        'beta' : beta,
        'gamma': gamma,
        'volume'         : volume,
        'volume_per_atom': float(volume_per_atom),
        'density'        : density,

        # Tỉ lệ và biến dạng mạng
        'b_over_a'          : float(b_over_a),
        'c_over_a'          : float(c_over_a),
        'angle_deviation'   : float(angle_deviation),
        'angle_std'         : angle_std,           # [MỚI]
        'lattice_anisotropy': float(lattice_anisotropy),  # [MỚI]

        # Đối xứng
        'space_group_number': space_group_number,
        'space_group_symbol': space_group_symbol,
        'crystal_system'    : crystal_system,
        'symprec_used'      : symprec_used,        # [MỚI]

        # Liên kết
        'mean_bond_length' : mean_bond_length,
        'std_bond_length'  : std_bond_length,
        'min_bond_length'  : min_bond_length,
        'max_bond_length'  : max_bond_length,
        'bond_length_range': bond_length_range,    # [MỚI]

        # Coordination
        'mean_coordination': mean_coordination,
        'std_coordination' : std_coordination,
        'min_coordination' : min_coordination,
        'max_coordination' : max_coordination,

        # Hybridization [MỚI]
        'fraction_sp'   : fraction_sp,
        'fraction_sp2'  : fraction_sp2,
        'fraction_sp3'  : fraction_sp3,
        'fraction_other': fraction_other,

        # Packing [MỚI]
        'packing_fraction': float(packing_fraction),
    }

## 8. Kiểm tra hàm trên 5 mẫu

In [ ]:
test_rows = []
for idx, row in carbon_df.head(5).iterrows():
    feat = extract_features_v2(row[cif_col])
    feat['row_index']   = idx
    feat['split']       = row['split']
    feat['material_id'] = row.get('material_id', np.nan)
    feat['energy']      = row[energy_col] if energy_col else np.nan
    test_rows.append(feat)

test_df = pd.DataFrame(test_rows)
print('Columns:', test_df.columns.tolist())
display(test_df[[
    'material_id', 'num_atoms', 'density', 'crystal_system',
    'lattice_anisotropy', 'angle_std', 'bond_length_range',
    'fraction_sp', 'fraction_sp2', 'fraction_sp3',
    'packing_fraction', 'symprec_used'
]])

## 9. Trích xuất toàn bộ dữ liệu

In [ ]:
feature_rows = []
failed_rows  = []

CHECKPOINT_EVERY = 500
partial_path = f'{CHECKPT_DIR}/carbon24_features_v2_partial.csv'
failed_path  = f'{CHECKPT_DIR}/carbon24_failed_rows_v2.csv'

for idx, row in tqdm(carbon_df.iterrows(), total=len(carbon_df), desc='Extracting'):
    try:
        feat = extract_features_v2(row[cif_col])

        feat['row_index']   = idx
        feat['split']       = row.get('split', 'unknown')
        feat['source_file'] = row.get('source_file', 'unknown')

        for id_col in ['material_id', 'id', 'mp_id', 'jid', 'name']:
            if id_col in carbon_df.columns:
                feat[id_col] = row[id_col]

        if energy_col and energy_col in carbon_df.columns:
            feat['energy'] = row[energy_col]

        feature_rows.append(feat)

    except Exception as e:
        failed_rows.append({'row_index': idx, 'error': str(e)})

    if (idx + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(feature_rows).to_csv(partial_path, index=False)
        pd.DataFrame(failed_rows).to_csv(failed_path, index=False)
        print(f'  Checkpoint {idx+1}: ok={len(feature_rows)}, failed={len(failed_rows)}')

feat_df   = pd.DataFrame(feature_rows)
failed_df = pd.DataFrame(failed_rows)

print(f'\nDone. Features: {feat_df.shape}, Failed: {failed_df.shape}')
display(feat_df.head(3))

## 10. Tính `relative_energy` — chỉ dùng min của tập train

> **Lý do:** Tính `min()` trên toàn bộ dataset (gộp train+val+test) gây **data leakage** khi dùng cho bài toán dự đoán năng lượng. Phiên bản này tính baseline từ tập train rồi áp dụng cho tất cả.

In [ ]:
if 'energy' in feat_df.columns:
    feat_df['energy'] = pd.to_numeric(feat_df['energy'], errors='coerce')

    # Baseline = min energy của tập TRAIN
    train_min_energy = feat_df.loc[feat_df['split'] == 'train', 'energy'].min()
    print(f'Train min energy : {train_min_energy:.6f} eV/atom')
    print(f'Global min energy: {feat_df["energy"].min():.6f} eV/atom')

    feat_df['relative_energy'] = feat_df['energy'] - train_min_energy

    print('\nRelative energy stats:')
    print(feat_df.groupby('split')['relative_energy'].describe().round(4))
else:
    print('Không tìm thấy cột energy.')

## 11. Kiểm tra chất lượng đặc trưng

In [ ]:
# Missing values
missing = feat_df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if missing.empty:
    print('Không có missing values.')
else:
    print('Missing values:')
    print(missing)
    print()
    missing_report = missing.reset_index()
    missing_report.columns = ['column', 'missing_count']
    missing_report['missing_ratio'] = missing_report['missing_count'] / len(feat_df)
    display(missing_report)

In [ ]:
# Phân phối hybridization
print('=== Phân phối hybridization (fraction trung bình theo split) ===')
hybrid_cols = ['fraction_sp', 'fraction_sp2', 'fraction_sp3', 'fraction_other']
print(feat_df.groupby('split')[hybrid_cols].mean().round(4))

print('\n=== Phân phối hybridization (toàn bộ dataset) ===')
print(feat_df[hybrid_cols].describe().round(4))

In [ ]:
# Kiểm tra symprec_used — bao nhiêu cấu trúc cần fallback về loose?
if 'symprec_used' in feat_df.columns:
    sp_counts = feat_df['symprec_used'].value_counts(dropna=False)
    print('symprec_used distribution:')
    print(sp_counts)
    pct_loose = (feat_df['symprec_used'] == 0.1).sum() / len(feat_df) * 100
    print(f'\n{pct_loose:.1f}% cấu trúc cần fallback symprec=0.1')

In [ ]:
# Thống kê tổng quan
print('=== Thống kê đặc trưng số ===')
numeric_cols = feat_df.select_dtypes(include=np.number).columns.tolist()
display(feat_df[numeric_cols].describe().T.round(4))

## 12. Lưu file

In [ ]:
features_path      = f'{DATA_DIR}/carbon24_features_v2.csv'
failed_final_path  = f'{DATA_DIR}/carbon24_failed_rows_v2.csv'
missing_report_path = f'{DATA_DIR}/carbon24_missing_report_v2.csv'

feat_df.to_csv(features_path, index=False)
failed_df.to_csv(failed_final_path, index=False)

# Lưu missing report
missing_full = feat_df.isna().sum().reset_index()
missing_full.columns = ['column', 'missing_count']
missing_full['missing_ratio'] = missing_full['missing_count'] / len(feat_df)
missing_full.to_csv(missing_report_path, index=False)

print(f'Features saved  : {features_path}')
print(f'Failed rows     : {failed_final_path}')
print(f'Missing report  : {missing_report_path}')
print(f'\nShape: {feat_df.shape}')
print(f'Columns ({len(feat_df.columns)}): {feat_df.columns.tolist()}')

In [ ]:
%cd /kaggle/working

!zip -r carbon24_features_v2.zip \
    carbon24_project_v2/data/carbon24_features_v2.csv \
    carbon24_project_v2/data/carbon24_failed_rows_v2.csv \
    carbon24_project_v2/data/carbon24_missing_report_v2.csv

!ls -lh carbon24_features_v2.zip